In [ ]:
import argparse
import gzip
import pickle
import matplotlib.pyplot as plt
import yaml
import io
import os
import numpy as np
import hist
from typing import Any, IO, Dict, List, Iterable
import dctools
from dctools import plot as plotter
from dctools.plot import plotting
import matplotlib.pyplot as plt
import scipy.interpolate as interp
import mplhep as hep
from scipy import stats as st
np.seterr(all='warn')
plt.ioff()
from dctools import dict_to_hist_axis


In [ ]:
def plotting(config, variable, channel, rebin=1, xlim=[], blind=False, era="someyear", checksyst=True,
             remap_replacement_types = None, logy=True, logx=False, bin_width_norm=None, no_ratios=False,
             combine_fit="pre-combine", combine_total_uncertainty="total_background", combine_channel_group=None) -> None:
    assert combine_fit in ["pre-combine", "prefit", "fit_b", "fit_s"]
    if remap_replacement_types is None:
        remap_replacement_types = [] #expected args: "datadriven", "validation"
    datasets:Dict = dict()
    color_cycle:List = []
    edges:Iterable[str] | Iterable[float] | None = None
    combine_uncertainty_histo: hist.Hist | None = None
    combine_channels: List[str] | None = None
    combine_lumi: int | None = None
    combine_era: str | None = None

    if bin_width_norm is None and "bin_width_norm" in config:
        bin_width_norm = config.bin_width_norm

    for ng, name in enumerate(config.groups):
            histograms = dict(
                filter(
                    lambda _n: _n[0] in config.groups[name].processes,
                    config.boosthist.items()
                )
            )
            p = dctools.datagroup(
                histograms       = histograms,
                ptype            = config.groups[name].type,
                observable       = variable,
                name             = name,
                xsections        = config.xsections,
                channel          = channel,
                luminosity       = config.luminosity.value,
                rebin            = rebin,
                remap_class_name = config.groups[name].remap_class_name if "remap_class_name" in config.groups[name] else None,
            )
           
            if p.remap_replace_group_name is not None:
                if p.remap_replace_type in remap_replacement_types:
                    print(f"Overwriting: channel: {p.channel} type: {p.remap_replace_type}, {p.remap_replace_group_name} replaced by {p.name}")
                    # overwrite a previously defined dataset in the dictionary. This requires the remap types to be after ALL MC in the config file (and still before the real data)
                    datasets[p.remap_replace_group_name] = p
                    if hasattr(config.groups[name], "color") and len(p.to_boost().shape):
                        # must replace the previous color cycler...
                        index = list(datasets.keys()).index(p.remap_replace_group_name)
                        color_cycle[index] = config.groups[name].color
                else:
                    print(f"Skipping: channel: {p.channel} type: {p.remap_replace_type}, {p.remap_replace_group_name} would have been replaced by {p.name}")
                    # this process is ignored / not added to the stack
                    continue
            else:
                datasets[p.name] = p
                if hasattr(config.groups[name], "color") and len(p.to_boost().shape):
                    color_cycle.append(config.groups[name].color)
            if p.ptype == "signal":
                signal = p.name
    _plot_channel = plotter.add_process_axis(datasets)
    variable_in_axes = variable if variable in _plot_channel.axes.name else ""
    pred = _plot_channel.project('process', 'systematic', variable_in_axes)[:hist.loc('data'),:,:]
    data = _plot_channel[{'systematic':'nominal'}].project('process', variable_in_axes)[hist.loc('data'),:]
    print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('data'),:])
    print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WZ_ewk'),:])
    print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('DY'),:])
    print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WW'),:])
    print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('ZZ'),:])
    print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WZ'),:])
    print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('Top'),:])
    print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('VVV'),:])
    print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('VBFZ'),:])
    # print("data = ", _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('data'),:].sum())
    # print("WZ_ewk = " ,_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WZ_ewk'),:].sum())
    # print("DY = ", _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('DY'),:].sum())
    # print("WW = ", _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WW'),:].sum())
    # print("ZZ = ", _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('ZZ'),:].sum())
    # print("WZ = ", _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WZ'),:].sum())
    # print("Top = ", _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('Top'),:].sum())
    # print("VVV = ", _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('VVV'),:].sum())
    # # print("ggVV = ", _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('ggVV'),:].sum())
    # print("VBFZ = ", _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('VBFZ'),:].sum())
    
    
    
    plt.figure(figsize=(6, 4.9 if no_ratios else 7))
    ax, bx = plotter.mcplot(
        pred[{'systematic':'nominal'}].stack('process'),
        data=None if blind else data,
        syst=pred.stack('process'),
        colors = color_cycle,
        no_ratios=no_ratios,
        bin_width_norm=bin_width_norm,
        combine_fit=combine_fit,
        combine_uncertainty_histo=combine_uncertainty_histo[{'systematic':'nominal'}] if combine_uncertainty_histo else None,
        combine_histo_edges=edges,
    )
    
    ymax = np.max([10000]+[c.get_height() for c in ax.containers[0] if ~np.isnan(c.get_height())])
    ymin = np.min([0.001]+[c.get_height() for c in ax.containers[0] if ~np.isnan(c.get_height())])
    
    ax.set_ylim(0.001, 100*ymax)
    #changes made for grant proposal plots
    # ax.set_ylim(0.001, 1200)
    try:
        sig_ewk = _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WZ_ewk'),:]   
        sig_qcd = _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WZ'),:]   
        sig_ewk.plot(ax=ax, histtype='step', color='red', binwnorm=bin_width_norm)
        sig_qcd.plot(ax=ax, histtype='step', color='purple', binwnorm=bin_width_norm)
    except:
        pass
    if bx is not None:
        bx.set_ylim([0.1, 1.9])
        if len(xlim) > 0:
            bx.set_xlim(xlim)
    elif len(xlim) > 0:
        ax.set_xlim(xlim)
    ax.set_title(f"channel {combine_channel_group or channel}: {combine_era or era}")
    hep.cms.label("", ax=ax, data=not blind, lumi=combine_lumi, year=combine_era or int(era)) #add lumi=lumi, add year=int(era) with handling of APV, etc.
    
    if logy :
        ylim_orig = ax.get_ylim()
        if ylim_orig[0]<=0:
            ax.set_ylim(10^(-1), ylim_orig[1])
        ax.set_yscale('log')
    if logx :
        xlim_orig = ax.get_xlim()
        if xlim_orig[0]<=0:
            ax.set_xlim(10**(1.5), xlim_orig[1])
            # print(xlim_orig[1])
        ax.set_xscale('log')
    # ax.set_yscale('log')
    # ax.set_xlim(50,1500)
    # ax.set_xscale('log')

    # if checksyst:
    #     pred = _plot_channel.project('process','systematic', variable)[:hist.loc('data'),:,:]
    #     data = _plot_channel[{'systematic':'nominal'}].project('process',variable)[hist.loc('data'),:] 
    #     plotter.check_systematic(
    #         pred[{'systematic':'nominal'}].stack('process'),
    #         syst=pred.stack('process'),
    #         plot_file_name=f'check-sys-{channel}-{era}', 
    #         xrange=xlim,
    #         no_ratios=no_ratios,
    #         bin_width_norm=bin_width_norm,
    #     )
    #     plt.clf()
    return _plot_channel, datasets


        

In [ ]:
config_2018 = dctools.read_config("config/inc-WZ/input_UL_2018-WZ_inclusive.yaml")
y = "2018"
c = "inc-SR1"
v = "dilep_tau_loose_met_hadron_mt"
# v = 'tau_pt_loose'
rrt = "datadriven"

for channel in config_2018.plotting:
    ch_cfg = config_2018.plotting[channel]
    if (c not in channel): continue
    for vname in ch_cfg:
        if v not in vname: continue
        v_cfg = ch_cfg[vname]
        plotting(config_2018, vname, channel,
             rebin = v_cfg.rebin,
             xlim = [],
             blind=v_cfg.blind,
             era = "2018",
             remap_replacement_types = rrt,
             #logx="logx",
             #bin_width_norm = "global_bin_width_norm",
             #no_ratios = "no_ratios",
             # checksyst = True,
             # combine_total_uncertainty = "combine_total_uncertainty",
             # combine_channel_group = None,
            )

In [ ]:
config_2018 = dctools.read_config("config/inc-WZ/input_UL_2018-WZ_inclusive.yaml")
y = "2018"
c = "inc-SR1l"
# v = "dilep_tau_loose_met_hadron_mt"
v = 'njets'
# rrt = "datadriven"

for channel in config_2018.plotting:
    ch_cfg = config_2018.plotting[channel]
    if (c not in channel): continue
    for vname in ch_cfg:
        if v not in vname: continue
        v_cfg = ch_cfg[vname]
        plotting(config_2018, vname, channel,
             rebin = v_cfg.rebin,
             xlim = [],
             blind=v_cfg.blind,
             era = "2018",
             # remap_replacement_types = rrt,
             #logx="logx",
             #bin_width_norm = "global_bin_width_norm",
             #no_ratios = "no_ratios",
             # checksyst = True,
             # combine_total_uncertainty = "combine_total_uncertainty",
             # combine_channel_group = None,
            )

In [ ]:
config_2018 = dctools.read_config("config/inc-WZ/input_UL_2018-WZ_inclusive.yaml")
y = "2018"
c = "inc-DY0"
v = "dilep_tau_loose_met_hadron_mt"
# v = 'met_pt'
# rrt = "datadriven"

for channel in config_2018.plotting:
    ch_cfg = config_2018.plotting[channel]
    if (c not in channel): continue
    for vname in ch_cfg:
        if v not in vname: continue
        v_cfg = ch_cfg[vname]
        plotting(config_2018, vname, channel,
             rebin = v_cfg.rebin,
             xlim = [],
             blind=v_cfg.blind,
             era = "2018",
             # remap_replacement_types = rrt,
             # logx="logx",
             #bin_width_norm = "global_bin_width_norm",
             #no_ratios = "no_ratios",
             # checksyst = True,
             # combine_total_uncertainty = "combine_total_uncertainty",
             # combine_channel_group = None,
            )

In [ ]:
config_2018 = dctools.read_config("config/inc-WZ/input_UL_2018-WZ_inclusive.yaml")
y = "2018"
c = "inc-DY1"
v = "dilep_tau_loose_met_hadron_mt"
# v = 'dilep_pt'
rrt = "datadriven"

for channel in config_2018.plotting:
    ch_cfg = config_2018.plotting[channel]
    if (c not in channel): continue
    for vname in ch_cfg:
        if v not in vname: continue
        v_cfg = ch_cfg[vname]
        plotting(config_2018, vname, channel,
             rebin = v_cfg.rebin,
             xlim = [],
             blind=v_cfg.blind,
             era = "2018",
             remap_replacement_types = rrt,
             #logx="logx",
             #bin_width_norm = "global_bin_width_norm",
             #no_ratios = "no_ratios",
             # checksyst = True,
             # combine_total_uncertainty = "combine_total_uncertainty",
             # combine_channel_group = None,
            )

In [ ]:
def plotting(config, variable, channel, rebin=1, xlim=[], blind=False, era="someyear", checksyst=True,
             remap_replacement_types = None, logy=True, logx=False, bin_width_norm=None, no_ratios=False,
             combine_fit="pre-combine", combine_total_uncertainty="total_background", combine_channel_group=None) -> None:
    assert combine_fit in ["pre-combine", "prefit", "fit_b", "fit_s"]
    if remap_replacement_types is None:
        remap_replacement_types = [] #expected args: "datadriven", "validation"
    datasets:Dict = dict()
    color_cycle:List = []
    edges:Iterable[str] | Iterable[float] | None = None
    combine_uncertainty_histo: hist.Hist | None = None
    combine_channels: List[str] | None = None
    combine_lumi: int | None = None
    combine_era: str | None = None

    if bin_width_norm is None and "bin_width_norm" in config:
        bin_width_norm = config.bin_width_norm

    for ng, name in enumerate(config.groups):
            histograms = dict(
                filter(
                    lambda _n: _n[0] in config.groups[name].processes,
                    config.boosthist.items()
                )
            )
            p = dctools.datagroup(
                histograms       = histograms,
                ptype            = config.groups[name].type,
                observable       = variable,
                name             = name,
                xsections        = config.xsections,
                channel          = channel,
                luminosity       = config.luminosity.value,
                rebin            = rebin,
                remap_class_name = config.groups[name].remap_class_name if "remap_class_name" in config.groups[name] else None,
            )
           
            if p.remap_replace_group_name is not None:
                if p.remap_replace_type in remap_replacement_types:
                    print(f"Overwriting: channel: {p.channel} type: {p.remap_replace_type}, {p.remap_replace_group_name} replaced by {p.name}")
                    # overwrite a previously defined dataset in the dictionary. This requires the remap types to be after ALL MC in the config file (and still before the real data)
                    datasets[p.remap_replace_group_name] = p
                    if hasattr(config.groups[name], "color") and len(p.to_boost().shape):
                        # must replace the previous color cycler...
                        index = list(datasets.keys()).index(p.remap_replace_group_name)
                        color_cycle[index] = config.groups[name].color
                else:
                    print(f"Skipping: channel: {p.channel} type: {p.remap_replace_type}, {p.remap_replace_group_name} would have been replaced by {p.name}")
                    # this process is ignored / not added to the stack
                    continue
            else:
                datasets[p.name] = p
                if hasattr(config.groups[name], "color") and len(p.to_boost().shape):
                    color_cycle.append(config.groups[name].color)
            if p.ptype == "signal":
                signal = p.name
    _plot_channel = plotter.add_process_axis(datasets)
    variable_in_axes = variable if variable in _plot_channel.axes.name else ""
    pred = _plot_channel.project('process', 'systematic', variable_in_axes)[:hist.loc('data'),:,:]
    data = _plot_channel[{'systematic':'nominal'}].project('process', variable_in_axes)[hist.loc('data'),:]
    # print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('data'),:])
    # print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WZ_ewk'),:])
    # print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('DY'),:])
    # print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WW'),:])
    # print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('ZZ'),:])
    # print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WZ'),:])
    # print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('Top'),:])
    # print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('VVV'),:])
    # print(_plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('VBFZ'),:])
    # data = _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('data'),:]
    WZ_ewk = _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WZ_ewk'),:]
    DY =  _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('DY'),:]
    WW =  _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WW'),:]
    ZZ =  _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('ZZ'),:]
    WZ =  _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WZ'),:]
    Top = _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('Top'),:]
    VVV = _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('VVV'),:]
    VBFZ = _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('VBFZ'),:]

    data_vals = data.view()
    WZ_ewk_vals = WZ_ewk.view()
    WW_vals = WW.view()
    ZZ_vals = ZZ.view()
    WZ_vals = WZ.view()
    Top_vals = Top.view()
    VVV_vals = VVV.view()
    VBFZ_vals = VBFZ.view()
    
    # Now compute DY component
    
    data_DY_ch = data_vals - (WZ_ewk_vals + WW_vals + ZZ_vals + WZ_vals + Top_vals + VVV_vals + VBFZ_vals)
    
    print(data_DY_ch)
    
    plt.figure(figsize=(6, 4.9 if no_ratios else 7))
    ax, bx = plotter.mcplot(
        pred[{'systematic':'nominal'}].stack('process'),
        data=None if blind else data,
        syst=pred.stack('process'),
        colors = color_cycle,
        no_ratios=no_ratios,
        bin_width_norm=bin_width_norm,
        combine_fit=combine_fit,
        combine_uncertainty_histo=combine_uncertainty_histo[{'systematic':'nominal'}] if combine_uncertainty_histo else None,
        combine_histo_edges=edges,
    )
    
    ymax = np.max([10000]+[c.get_height() for c in ax.containers[0] if ~np.isnan(c.get_height())])
    ymin = np.min([0.001]+[c.get_height() for c in ax.containers[0] if ~np.isnan(c.get_height())])
    
    ax.set_ylim(0.001, 100*ymax)
    #changes made for grant proposal plots
    # ax.set_ylim(0.001, 1200)
    try:
        sig_ewk = _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WZ_ewk'),:]   
        sig_qcd = _plot_channel[{'systematic':'nominal'}].project('process', variable)[hist.loc('WZ'),:]   
        sig_ewk.plot(ax=ax, histtype='step', color='red', binwnorm=bin_width_norm)
        sig_qcd.plot(ax=ax, histtype='step', color='purple', binwnorm=bin_width_norm)
    except:
        pass
    if bx is not None:
        bx.set_ylim([0.1, 1.9])
        if len(xlim) > 0:
            bx.set_xlim(xlim)
    elif len(xlim) > 0:
        ax.set_xlim(xlim)
    ax.set_title(f"channel {combine_channel_group or channel}: {combine_era or era}")
    # hep.cms.label("", ax=ax, data=not blind, lumi=combine_lumi, year=combine_era or int(era)) #add lumi=lumi, add year=int(era) with handling of APV, etc.
    hep.cms.label("", ax=ax, data=not blind, lumi=combine_lumi, year=era)
    
    if logy :
        ylim_orig = ax.get_ylim()
        if ylim_orig[0]<=0:
            ax.set_ylim(10^(-1), ylim_orig[1])
        ax.set_yscale('log')
    if logx :
        xlim_orig = ax.get_xlim()
        if xlim_orig[0]<=0:
            ax.set_xlim(10**(1.5), xlim_orig[1])
            # print(xlim_orig[1])
        ax.set_xscale('log')
    # ax.set_yscale('log')
    # ax.set_xlim(50,1500)
    # ax.set_xscale('log')

    # if checksyst:
    #     pred = _plot_channel.project('process','systematic', variable)[:hist.loc('data'),:,:]
    #     data = _plot_channel[{'systematic':'nominal'}].project('process',variable)[hist.loc('data'),:] 
    #     plotter.check_systematic(
    #         pred[{'systematic':'nominal'}].stack('process'),
    #         syst=pred.stack('process'),
    #         plot_file_name=f'check-sys-{channel}-{era}', 
    #         xrange=xlim,
    #         no_ratios=no_ratios,
    #         bin_width_norm=bin_width_norm,
    #     )
    #     plt.clf()
    return _plot_channel, datasets


        

In [ ]:
from boost_histogram import histogram
config_2016APV = dctools.read_config("config/inc-WZ/input_UL_2016APV-WZ_inclusive.yaml")
y = "2016APV"
c = "inc-C0"
# v = "dilep_tau_loose_met_hadron_mt"
v = 'tau_pt_loose'
# rrt = "validation"

for channel in config_2016APV.plotting:
    ch_cfg = config_2016APV.plotting[channel]
    if (c not in channel): continue
    for vname in ch_cfg:
        if v not in vname: continue
        v_cfg = ch_cfg[vname]
        plotting(config_2016APV, vname, channel,
             rebin = v_cfg.rebin,
             # xlim = [],
             blind=v_cfg.blind,
             era = "2016APV",
             # remap_replacement_types = rrt,
             # logx="logx",
             #bin_width_norm = "global_bin_width_norm",
             #no_ratios = "no_ratios",
             # checksyst = True,
             # combine_total_uncertainty = "combine_total_uncertainty",
             # combine_channel_group = None,
            )

In [ ]:
from boost_histogram import histogram
config_2017 = dctools.read_config("config/inc-WZ/input_UL_2017-WZ_inclusive.yaml")
y = "2017"
c = "inc-D1"
# v = "dilep_tau_loose_met_hadron_mt"
v = 'tau_pt'
# rrt = "validation"

for channel in config_2017.plotting:
    ch_cfg = config_2017.plotting[channel]
    if (c not in channel): continue
    for vname in ch_cfg:
        if v not in vname: continue
        v_cfg = ch_cfg[vname]
        plotting(config_2017, vname, channel,
             rebin = v_cfg.rebin,
             xlim = [],
             blind=v_cfg.blind,
             era = "2017",
             # remap_replacement_types = rrt,
             # logx="logx",
             #bin_width_norm = "global_bin_width_norm",
             #no_ratios = "no_ratios",
             # checksyst = True,
             # combine_total_uncertainty = "combine_total_uncertainty",
             # combine_channel_group = None,
            )

In [ ]:
import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np

config_2018 = dctools.read_config("config/inc-WZ/input_UL_2018-WZ_inclusive.yaml")
y = "2018"
c = "inc-SR0"
v = "dilep_tau_loose_met_hadron_mt"
rrt = "datadriven"
for channel in config_2018.plotting:
    ch_cfg = config_2018.plotting[channel]
    if (c not in channel): continue
    for vname in ch_cfg:
        if v not in vname: continue
        v_cfg = ch_cfg[vname]
        _plot_channel, datasets = plotting(
            config_2018, v, c,
            rebin=[5]*17+[65],
            xlim=[],
            blind=v_cfg.blind,
            era="2018",
            remap_replacement_types=rrt,
        )

        h = _plot_channel[{'systematic': 'nominal'}].project('process', vname)
        
        h_WZ     = h[hist.loc('WZ'), :]
        h_DY     = h[hist.loc('DY'), :]
        h_WW     = h[hist.loc('WW'), :]
        h_WZewk  = h[hist.loc('WZ_ewk'), :]
        h_ZZ     = h[hist.loc('ZZ'), :]
        h_VBFZ   = h[hist.loc('VBFZ'), :]
        h_Top   = h[hist.loc('Top'), :]
        h_VVV   = h[hist.loc('VVV'), :]

        h_bkg = h_DY + h_WW + h_WZewk + h_ZZ + h_VBFZ + h_Top + h_VVV
        
        def safe_ratio(num, den):
            den_vals = den.values()
            num_vals = num.values()
            mask = den_vals == 0
            ratio_vals = np.zeros_like(num_vals, dtype=float)
            ratio_vals[~mask] = num_vals[~mask] / den_vals[~mask]
            ratio = num.copy()
            ratio.values()[...] = ratio_vals
            return ratio
        
        r_WZ_bkg    = safe_ratio(h_WZ, h_bkg)
        
        # --- plot ---
        plt.figure(figsize=(6, 4))
        ax = plt.gca()
        
        r_WZ_bkg.plot(ax=ax, histtype='step', label='Sig/bkg')
    
        
        ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
        ax.set_xlabel('$M_T^{WZ}$')
        ax.set_ylabel('Ratio')
        # ax.set_ylim(0, 5)
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # hep.cms.label("", ax=ax, year=2018)
        ax.set_title(f"Channel: {channel}")
        
        plt.tight_layout()
        plt.show()



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import hist
import dctools
from dctools import plot as plotter
from dctools.plot import plotting

config_2018 = dctools.read_config("config/inc-WZ/input_UL_2018-WZ_inclusive.yaml")


sig = "WZTo3LNu_TuneCP5_13TeV-amcatnloFXFX-pythia8"
bkg = ['DYJetsToLL_0J_TuneCP5_13TeV-amcatnloFXFX-pythia8', 
       'DYJetsToLL_1J_TuneCP5_13TeV-amcatnloFXFX-pythia8', 
       'DYJetsToLL_2J_TuneCP5_13TeV-amcatnloFXFX-pythia8', 
       'GluGluToWWToENEN_TuneCP5_13TeV_MCFM701_pythia8', 
       'GluGluToWWToENMN_TuneCP5_13TeV_MCFM701_pythia8', 
       'GluGluToWWToENTN_TuneCP5_13TeV_MCFM701_pythia8', 
       'GluGluToWWToMNMN_TuneCP5_13TeV_MCFM701_pythia8',
       'GluGluToWWToMNEN_TuneCP5_13TeV_MCFM701_pythia8',
       'GluGluToWWToMNTN_TuneCP5_13TeV_MCFM701_pythia8',
       'WZZ_TuneCP5_13TeV-amcatnlo-pythia8', 
       'GluGluToWWToTNMN_TuneCP5_13TeV_MCFM701_pythia8',
       'GluGluToWWToTNTN_TuneCP5_13TeV_MCFM701_pythia8', 
       'GluGluToContinToZZTo2e2mu_TuneCP5_13TeV-mcfm701-pythia8', 
       'GluGluToContinToZZTo2e2nu_TuneCP5_13TeV-mcfm701-pythia8', 
       'GluGluToContinToZZTo2e2tau_TuneCP5_13TeV-mcfm701-pythia8', 
       'GluGluToContinToZZTo2mu2nu_TuneCP5_13TeV-mcfm701-pythia8', 
       'GluGluToContinToZZTo2mu2tau_TuneCP5_13TeV-mcfm701-pythia8', 
       'GluGluToContinToZZTo4e_TuneCP5_13TeV-mcfm701-pythia8', 
       'GluGluToContinToZZTo4mu_TuneCP5_13TeV-mcfm701-pythia8', 
       'GluGluToContinToZZTo4tau_TuneCP5_13TeV-mcfm701-pythia8',
       'ST_tW_antitop_5f_inclusiveDecays_TuneCP5_13TeV-powheg-pythia8',
       'ST_tW_top_5f_inclusiveDecays_TuneCP5_13TeV-powheg-pythia8', 
       'ST_tW_top_5f_NoFullyHadronicDecays_TuneCP5_13TeV-powheg-pythia8', 
       'ST_tW_antitop_5f_NoFullyHadronicDecays_TuneCP5_13TeV-powheg-pythia8',
       'ST_s-channel_4f_leptonDecays_TuneCP5_13TeV-amcatnlo-pythia8', 
       'ST_t-channel_antitop_4f_InclusiveDecays_TuneCP5_13TeV-powheg-madspin-pythia8', 
       'ST_t-channel_top_4f_InclusiveDecays_TuneCP5_13TeV-powheg-madspin-pythia8', 
       'tZq_ll_4f_ckm_NLO_TuneCP5_13TeV-amcatnlo-pythia8', 
       'TTTo2L2Nu_TuneCP5_13TeV-powheg-pythia8', 
       'TTWJetsToLNu_TuneCP5_13TeV-amcatnloFXFX-madspin-pythia8', 
       'TTWJetsToQQ_TuneCP5_13TeV-amcatnloFXFX-madspin-pythia8', 
       'TTZToLLNuNu_M-10_TuneCP5_13TeV-amcatnlo-pythia8', 
       'TTZToQQ_TuneCP5_13TeV-amcatnlo-pythia8', 
       'TTToSemiLeptonic_TuneCP5_13TeV-powheg-pythia8', 
       'WWTo2L2Nu_TuneCP5_13TeV-powheg-pythia8', 
       'WZJJ_EWK_InclusivePolarization_TuneCP5_13TeV_madgraph-madspin-pythia8',
       'ZZTo2L2Nu_TuneCP5_13TeV_powheg_pythia8', 
       'ZZJJ_ZZTo2L2Nu_EWK_dipoleRecoil_TuneCP5_13TeV-madgraph-pythia8',
       'EWKZ2Jets_ZToLL_M-50_TuneCP5_withDipoleRecoil_13TeV-madgraph-pythia8', 
       'EWKWPlus2Jets_WToLNu_M-50_TuneCP5_withDipoleRecoil_13TeV-madgraph-pythia8', 
       'WWW_4F_TuneCP5_13TeV-amcatnlo-pythia8', 
       'EWKWMinus2Jets_WToLNu_M-50_TuneCP5_withDipoleRecoil_13TeV-madgraph-pythia8',
       'WWZ_4F_TuneCP5_13TeV-amcatnlo-pythia8', 
       'ZZZ_TuneCP5_13TeV-amcatnlo-pythia8', 
       'WJetsToLNu_TuneCP5_13TeV-madgraphMLM-pythia8', 
       'ZZTo4L_TuneCP5_13TeV_powheg_pythia8'
      ]
data = ['MuonEG', 'DoubleMuon', 'SingleMuon', 'EGamma']
h = config_2018.boosthist[sig]["hist"]["dilep_tau_loose_met_hadron_mt"]
h_DY = config_2018.boosthist['DYJetsToLL_0J_TuneCP5_13TeV-amcatnloFXFX-pythia8']["hist"]["dilep_tau_loose_met_hadron_mt"]
# print(h_DY[{"channel": "inc-SR0", "systematic": "nominal", "dilep_tau_loose_met_hadron_mt": hist.rebin(groups = [1]*30+[2]*10+[10]*10)}].sum())
h_bkg = None
h_data = None
for sample in bkg:
    h_tmp = (
        config_2018.boosthist[sample]["hist"]["dilep_tau_loose_met_hadron_mt"]
    )
    if h_bkg is None:
        h_bkg = h_tmp
    else:
        h_bkg = h_bkg + h_tmp

for sample in data:
    h_tmp = (
        config_2018.boosthist[sample]["hist"]["dilep_tau_loose_met_hadron_mt"]
    )
    if h_data is None:
        h_data = h_tmp
    else:
        h_data = h_data + h_tmp

# Projections
h_sig = h[{"channel": "inc-SR0", "systematic": "nominal", "dilep_tau_loose_met_hadron_mt": hist.rebin(1)}] #[1]*30+[2]*10+[10]*10
h_bkg = h_bkg[{"channel": "inc-SR0", "systematic": "nominal", "dilep_tau_loose_met_hadron_mt": hist.rebin(1)}]
h_data = h_data[{"channel": "inc-SR0", "systematic": "nominal", "dilep_tau_loose_met_hadron_mt": hist.rebin(1)}]
sig_vals = h_sig.values()
bkg_vals = h_bkg.values()
data_vals = h_data.values()
# print(bkg_vals)
# print(data_vals.sum())
# Bin centers for ratio plot
bin_edges = h_sig.axes["dilep_tau_loose_met_hadron_mt"].edges
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# # Ratios (safe division)
ratio = np.divide(sig_vals, bkg_vals, out=np.zeros_like(sig_vals), where=bkg_vals != 0)


# Figure with ratio panel
fig, (ax, rax) = plt.subplots(
    2, 1, figsize=(6, 7),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]}
)

# === Main plot ===
h_sig.plot(ax=ax, histtype="step", label="signal")
h_bkg.plot(ax=ax, histtype="step", label="background")


ax.set_ylabel("Events")
ax.set_yscale("log")
ax.set_title(r"$m_T^{WZ}$ for inc-SR0")
ax.legend()
ax.grid(True)

# === Ratio plot ===
rax.step(bin_centers, ratio, where="mid", label="Sig/Bkg", color="C1")
# rax.step(bin_centers, ratio_dn, where="mid", label="Down / Nom", color="C2")

# rax.axhline(1.0, color="black", linestyle="--", linewidth=1)
rax.set_ylabel("Ratio")
rax.set_xlabel(r"$m_T^{WZ}$ [GeV]")
# rax.set_ylim(-0.1, 2)
rax.legend()
rax.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import hist
import dctools
from dctools import plot as plotter
from dctools.plot import plotting

config_2018 = dctools.read_config("config/inc-WZ/input_UL_2018-WZ_inclusive.yaml")
# print(config_2018.boosthist.keys())
h = config_2018.boosthist["WZTo3LNu_TuneCP5_13TeV-amcatnloFXFX-pythia8"]["hist"]["dilep_tau_loose_met_hadron_mt"]
name = "WZTo3LNu_TuneCP5_13TeV-amcatnloFXFX-pythia8"
h['inc-SR0', 'nominal', :].project("dilep_tau_loose_met_hadron_mt").plot1d()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import hist
import dctools
from dctools import plot as plotter
from dctools.plot import plotting

config_2018 = dctools.read_config("config/inc-WZ/input_UL_2018-WZ_inclusive.yaml")
print(config_2018.boosthist.keys())

sig = "WZTo3LNu_TuneCP5_13TeV-amcatnloFXFX-pythia8"
bkg = ['DYJetsToLL_0J_TuneCP5_13TeV-amcatnloFXFX-pythia8', 'DYJetsToLL_1J_TuneCP5_13TeV-amcatnloFXFX-pythia8', 'DYJetsToLL_2J_TuneCP5_13TeV-amcatnloFXFX-pythia8', 'GluGluToWWToENEN_TuneCP5_13TeV_MCFM701_pythia8', 'GluGluToWWToENMN_TuneCP5_13TeV_MCFM701_pythia8', 'GluGluToWWToENTN_TuneCP5_13TeV_MCFM701_pythia8', 'GluGluToWWToMNMN_TuneCP5_13TeV_MCFM701_pythia8', 'GluGluToWWToMNEN_TuneCP5_13TeV_MCFM701_pythia8', 'GluGluToWWToMNTN_TuneCP5_13TeV_MCFM701_pythia8', 'WZZ_TuneCP5_13TeV-amcatnlo-pythia8', 'GluGluToWWToTNMN_TuneCP5_13TeV_MCFM701_pythia8', 'GluGluToWWToTNTN_TuneCP5_13TeV_MCFM701_pythia8', 'GluGluToContinToZZTo2e2mu_TuneCP5_13TeV-mcfm701-pythia8', 'GluGluToContinToZZTo2e2nu_TuneCP5_13TeV-mcfm701-pythia8', 'GluGluToContinToZZTo2e2tau_TuneCP5_13TeV-mcfm701-pythia8', 'GluGluToContinToZZTo2mu2nu_TuneCP5_13TeV-mcfm701-pythia8', 'GluGluToContinToZZTo2mu2tau_TuneCP5_13TeV-mcfm701-pythia8', 'GluGluToContinToZZTo4e_TuneCP5_13TeV-mcfm701-pythia8', 'GluGluToContinToZZTo4mu_TuneCP5_13TeV-mcfm701-pythia8', 'GluGluToContinToZZTo4tau_TuneCP5_13TeV-mcfm701-pythia8', 'ST_tW_antitop_5f_inclusiveDecays_TuneCP5_13TeV-powheg-pythia8', 'ST_tW_top_5f_inclusiveDecays_TuneCP5_13TeV-powheg-pythia8', 'ST_tW_top_5f_NoFullyHadronicDecays_TuneCP5_13TeV-powheg-pythia8', 'ST_tW_antitop_5f_NoFullyHadronicDecays_TuneCP5_13TeV-powheg-pythia8', 'ST_s-channel_4f_leptonDecays_TuneCP5_13TeV-amcatnlo-pythia8', 'ST_t-channel_antitop_4f_InclusiveDecays_TuneCP5_13TeV-powheg-madspin-pythia8', 'ST_t-channel_top_4f_InclusiveDecays_TuneCP5_13TeV-powheg-madspin-pythia8', 'tZq_ll_4f_ckm_NLO_TuneCP5_13TeV-amcatnlo-pythia8', 'TTTo2L2Nu_TuneCP5_13TeV-powheg-pythia8', 'TTWJetsToLNu_TuneCP5_13TeV-amcatnloFXFX-madspin-pythia8', 'TTWJetsToQQ_TuneCP5_13TeV-amcatnloFXFX-madspin-pythia8', 'TTZToLLNuNu_M-10_TuneCP5_13TeV-amcatnlo-pythia8', 'TTZToQQ_TuneCP5_13TeV-amcatnlo-pythia8', 'TTToSemiLeptonic_TuneCP5_13TeV-powheg-pythia8', 'WWTo2L2Nu_TuneCP5_13TeV-powheg-pythia8', 'WZJJ_EWK_InclusivePolarization_TuneCP5_13TeV_madgraph-madspin-pythia8', 'ZZTo2L2Nu_TuneCP5_13TeV_powheg_pythia8', 'ZZJJ_ZZTo2L2Nu_EWK_dipoleRecoil_TuneCP5_13TeV-madgraph-pythia8', 'EWKZ2Jets_ZToLL_M-50_TuneCP5_withDipoleRecoil_13TeV-madgraph-pythia8', 'EWKWPlus2Jets_WToLNu_M-50_TuneCP5_withDipoleRecoil_13TeV-madgraph-pythia8', 'WWW_4F_TuneCP5_13TeV-amcatnlo-pythia8', 'EWKWMinus2Jets_WToLNu_M-50_TuneCP5_withDipoleRecoil_13TeV-madgraph-pythia8', 'WWZ_4F_TuneCP5_13TeV-amcatnlo-pythia8', 'ZZZ_TuneCP5_13TeV-amcatnlo-pythia8', 'WJetsToLNu_TuneCP5_13TeV-madgraphMLM-pythia8', 'ZZTo4L_TuneCP5_13TeV_powheg_pythia8']
h = config_2018.boosthist[sig]["hist"]["dilep_tau_loose_met_hadron_mt"]
h_bkg = None

for sample in bkg:
    h_tmp = (
        config_2018.boosthist[sample]["hist"]["dilep_tau_loose_met_hadron_mt"]
    )
    if h_bkg is None:
        h_bkg = h_tmp
    else:
        h_bkg = h_bkg + h_tmp

# Projections
h_sig = h[{"channel": "inc-SR0", "systematic": "nominal", "dilep_tau_loose_met_hadron_mt": hist.rebin(groups = [5]*10+[20]*3+[40])}]
h_bkg = h_bkg[{"channel": "inc-SR0", "systematic": "nominal", "dilep_tau_loose_met_hadron_mt": hist.rebin(groups = [5]*10+[20]*3+[40])}]
sig_vals = h_sig.values()
bkg_vals = h_bkg.values()

# Bin centers for ratio plot
bin_edges = h_sig.axes["dilep_tau_loose_met_hadron_mt"].edges
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# # Ratios (safe division)
ratio = np.divide(sig_vals, bkg_vals, out=np.zeros_like(sig_vals), where=bkg_vals != 0)


# Figure with ratio panel
fig, (ax, rax) = plt.subplots(
    2, 1, figsize=(6, 7),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]}
)

# === Main plot ===
h_sig.plot(ax=ax, histtype="step", label="signal")
h_bkg.plot(ax=ax, histtype="step", label="background")


ax.set_ylabel("Events")
ax.set_yscale("log")
ax.set_title(r"$m_T^{WZ}$ for inc-SR0")
ax.legend()
ax.grid(True)

# === Ratio plot ===
rax.step(bin_centers, ratio, where="mid", label="Sig/Bkg", color="C1")
# rax.step(bin_centers, ratio_dn, where="mid", label="Down / Nom", color="C2")

# rax.axhline(1.0, color="black", linestyle="--", linewidth=1)
rax.set_ylabel("Ratio")
rax.set_xlabel(r"$m_T^{WZ}$ [GeV]")
rax.set_ylim(-0.1, 2)
rax.legend()
rax.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import hist
import dctools
from dctools import plot as plotter
from dctools.plot import plotting

config_2018 = dctools.read_config("config/inc-WZ/input_UL_2018-WZ_inclusive.yaml")


name = "WZTo3LNu_TuneCP5_13TeV-amcatnloFXFX-pythia8"
h = config_2018.boosthist[name]["hist"]["dilep_tau_loose_met_hadron_mt"]

s = ["nominal", "UESUp", "UESDown"]


# Projections
h_nom = h[{"channel": "inc-SR0", "systematic": s[0], "dilep_tau_loose_met_hadron_mt": hist.rebin(groups=[30]*3+[60])}]
h_up  = h[{"channel": "inc-SR0", "systematic": s[1], "dilep_tau_loose_met_hadron_mt": hist.rebin(groups=[30]*3+[60])}]
h_dn  = h[{"channel": "inc-SR0", "systematic": s[2], "dilep_tau_loose_met_hadron_mt": hist.rebin(groups=[30]*3+[60])}]

nom_vals = h_nom.values()
up_vals  = h_up.values()
dn_vals  = h_dn.values()

# Bin centers for ratio plot
bin_edges = h_nom.axes["dilep_tau_loose_met_hadron_mt"].edges
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# Ratios (safe division)
ratio_up = np.divide(up_vals, nom_vals, out=np.zeros_like(up_vals), where=nom_vals != 0)
ratio_dn = np.divide(dn_vals, nom_vals, out=np.zeros_like(dn_vals), where=nom_vals != 0)

# Figure with ratio panel
fig, (ax, rax) = plt.subplots(
    2, 1, figsize=(6, 7),
    sharex=True,
    gridspec_kw={"height_ratios": [3, 1]}
)

# === Main plot ===
h_nom.plot(ax=ax, histtype="step", label=s[0])
h_up.plot(ax=ax, histtype="step", label=s[1])
h_dn.plot(ax=ax, histtype="step", label=s[2])

ax.set_ylabel("Events")
ax.set_yscale("log")
ax.set_title(r"$m_T^{WZ}$ for inc-SR0")
ax.legend()
ax.grid(True)

# === Ratio plot ===
rax.step(bin_centers, ratio_up, where="mid", label="Up / Nom", color="C1")
rax.step(bin_centers, ratio_dn, where="mid", label="Down / Nom", color="C2")

rax.axhline(1.0, color="black", linestyle="--", linewidth=1)
rax.set_ylabel("Ratio")
rax.set_xlabel(r"$m_T^{WZ}$ [GeV]")
rax.set_ylim(0.8, 1.2)
rax.legend()
rax.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import hist

name = "WZTo3LNu_TuneCP5_13TeV-amcatnloFXFX-pythia8"
h = config_2018.boosthist[name]["hist"]["mT_WZ"]

# Choose the systematics you want
systematics = ["nominal", "JERUp", "JERDown"]

# Create figure
plt.figure(figsize=(6, 5))

for s in systematics:
    h_proj = h[{
        "channel": "inc-SR0",
        "systematic": s,
        "mT_WZ": hist.rebin(groups=[2]*20+[5]*4)
    }]
    h_proj.plot(histtype='step', label=s)

plt.xlabel(r"$mT\_{WZ}$ [GeV]")
plt.ylabel("Events")
plt.title("m_T for inc-SR0: Nominal vs JERUp")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.yscale("log")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import hist

name = "WZTo3LNu_TuneCP5_13TeV-amcatnloFXFX-pythia8"
h = config_2018.boosthist[name]["hist"]["mT_WZ"]

# Choose the systematics you want
systematics = ["nominal", "QCDScale0wUp", "QCDScale0wDown"]

# Create figure
plt.figure(figsize=(6, 5))

for s in systematics:
    h_proj = h[{
        "channel": "inc-DY1",
        "systematic": s,
        "mT_WZ": hist.rebin(groups=[1]*30+[2]*5+[10]*2)
    }]
    h_proj.plot(histtype='step', label=s)

plt.xlabel(r"$mT\_WZ$ [GeV]")
plt.ylabel("Events")
plt.title("mT_WZ for inc-DY1: Nominal vs QCDScale0w")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
datasets:Dict = dict()
color_cycle:List = []
config =config_2018
a = []
for name in config.groups:
    histograms = dict(
        filter(
            lambda n: n[0] in config.groups[name].processes,
            config.boosthist.items()
            
        )
    )
    a.append(histograms)    
    
  #  p = dctools.datagroup(
   #     histograms = histograms,
    #    ptype      = config.groups[name].type,
     #   observable = 'tau_pt',
      #  name       = name,
       # xsections  = config.xsections,
        #channel    = 'vbs-Faketau_C',
        #luminosity = config.luminosity.value,
        #rebin      = 1
    #)
    #print(name)   
    #datasets[p.name] = p
    #if p.ptype == "Signal":
        #signal = p.name

#print(datasets)
#_plot_channel = plotter.add_process_axis(datasets)

#print(a)

In [ ]:
config_2018 = dctools.read_config("config/input_UL_2018-WZ_inclusive.yaml")
#name = "DYJetsToLL_M-50_TuneCP5_13TeV-madgraphMLM-pythia8"
name = "WZTo3LNu_TuneCP5_13TeV-amcatnloFXFX-pythia8"

In [ ]:
h = config_2018.boosthist[name]["hist"]["mT_WZ"]

In [ ]:
h['inc-D', 'nominal', :].project("mT_WZ").plot1d()

In [ ]:
h['inc-SR1', 'nominal', :].project("emu_mT_WZ").plot1d()

In [ ]:
h['inc-DY0', 'nominal', :].project("emu_mT_WZ").plot1d()

In [ ]:
h['inc-DY1', 'nominal', :].project("emu_mT_WZ").plot1d()

In [ ]:
h['inc-DY', 'nominal', :].project("baseweight").values(flow=True)

In [ ]:
signame="WZTo3LNu_TuneCP5_13TeV-amcatnloFXFX-pythia8"
vals = config_2018.boosthist[signame]["hist"]["emu_mT_WZ"][:, 'nominal', :].project("emu_mT_WZ").values(flow=True)


In [ ]:
len(vals)

In [ ]:
np.sum(vals[:200]), np.sum(vals[200:])